In [1]:

from pathlib import Path
import csv

# Define the project and raw-data paths
PROJECT_DIR = Path.cwd()

# If Jupyter opened in the notebooks folder, move to the project root
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DIR = PROJECT_DIR / "data" / "raw"

# Check that the sales file exists
input_file = RAW_DIR / "sales_train_validation.csv"

print("Project folder:", PROJECT_DIR)
print("Raw data folder:", RAW_DIR)
print("Sales file exists:", input_file.exists())

Project folder: c:\Users\babyp\Project-3-Retail-Demand-Forecasting-Inventory-Optimization
Raw data folder: c:\Users\babyp\Project-3-Retail-Demand-Forecasting-Inventory-Optimization\data\raw
Sales file exists: True


In [2]:

# Split the sales CSV into files smaller than 100 MB

chunks_dir = RAW_DIR / "sales_chunks"
chunks_dir.mkdir(parents=True, exist_ok=True)

max_bytes = 80 * 1024 * 1024  # 80 MiB per part
part_number = 0
current_bytes = 0
output_file = None
writer = None

with input_file.open("r", newline="", encoding="utf-8") as source:
    reader = csv.reader(source)
    header = next(reader)

    try:
        for row in reader:
            if writer is None or current_bytes >= max_bytes:
                if output_file is not None:
                    output_file.close()

                part_number += 1
                part_path = chunks_dir / f"sales_part_{part_number:02d}.csv"
                output_file = part_path.open(
                    "w", newline="", encoding="utf-8"
                )
                writer = csv.writer(output_file)
                writer.writerow(header)
                current_bytes = output_file.tell()

            writer.writerow(row)
            current_bytes = output_file.tell()
    finally:
        if output_file is not None:
            output_file.close()

print("Splitting completed!")
print("Number of parts:", part_number)

for part in sorted(chunks_dir.glob("sales_part_*.csv")):
    print(f"{part.name}: {part.stat().st_size / (1024 * 1024):.2f} MiB")

Splitting completed!
Number of parts: 2
sales_part_01.csv: 80.00 MiB
sales_part_02.csv: 34.49 MiB


In [3]:

from pathlib import Path
import csv

# Find the project folder
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DIR = PROJECT_DIR / "data" / "raw"

# Check the price file
prices_file = RAW_DIR / "sell_prices.csv"

print("Current folder:", PROJECT_DIR)
print("Price file exists:", prices_file.exists())
print("CSV module loaded:", csv.__name__)

Current folder: c:\Users\babyp\Project-3-Retail-Demand-Forecasting-Inventory-Optimization
Price file exists: True
CSV module loaded: csv


In [4]:

# Split sell_prices.csv into smaller CSV files for BigQuery

prices_file = RAW_DIR / "sell_prices.csv"
prices_chunks_dir = RAW_DIR / "sell_prices_chunks"
prices_chunks_dir.mkdir(parents=True, exist_ok=True)

max_bytes = 80 * 1024 * 1024  # 80 MiB per part
part_number = 0
current_bytes = 0
output_file = None
writer = None

with prices_file.open("r", newline="", encoding="utf-8") as source:
    reader = csv.reader(source)
    header = next(reader)

    try:
        for row in reader:
            if writer is None or current_bytes >= max_bytes:
                if output_file is not None:
                    output_file.close()

                part_number += 1
                part_path = (
                    prices_chunks_dir / f"sell_prices_part_{part_number:02d}.csv"
                )
                output_file = part_path.open(
                    "w", newline="", encoding="utf-8"
                )
                writer = csv.writer(output_file)
                writer.writerow(header)
                current_bytes = output_file.tell()

            writer.writerow(row)
            current_bytes = output_file.tell()
    finally:
        if output_file is not None:
            output_file.close()

print("Price-file splitting completed!")
print("Number of parts:", part_number)

for part in sorted(prices_chunks_dir.glob("sell_prices_part_*.csv")):
    print(f"{part.name}: {part.stat().st_size / (1024 * 1024):.2f} MiB")

Price-file splitting completed!
Number of parts: 3
sell_prices_part_01.csv: 80.00 MiB
sell_prices_part_02.csv: 80.00 MiB
sell_prices_part_03.csv: 40.50 MiB
